# Exploring CaImAn's output components and manual curation methods

Fits CNMF on the motion-corrected calcium movie and works through CaImAn's built-in methods for evaluating, visualizing, and manually curating the resulting components (`cnm.estimates`), as a hands-on reference before building a custom evaluation/adjustment tool on top of them.

Prerequisite: run `pipeline_smoke_test.ipynb` first so `CaData_Moco.h5` exists in `data/processed/2026_05_19_animal_1/`.

In [1]:
from pathlib import Path
import caiman as cm
from caiman.source_extraction.cnmf.cnmf import load_CNMF
import bokeh.io
bokeh.io.output_notebook()

from caiman_wrapper.motion_correction import start_cluster
from caiman_wrapper.source_extraction import build_cnmf_params, run_cnmf

Loading BokehJS ...

In [2]:
output_dir = Path('../data/processed/2026_05_19_animal_1')
ca_moco_path = output_dir / 'CaData_Moco.h5'

images = cm.load(str(ca_moco_path), is3D=True)
images.shape  # (T, Y, X, Z)

(930, 128, 128, 26)

In [3]:
c, dview, n_processes = start_cluster()

## Fit CNMF



In [7]:
import numpy as np
# Memory map motion corrected calcium data to prepare for analysis
ca_name_new = cm.save_memmap([images],base_name='memmap_', order='C',
                           border_to_0=0, dview=dview)
# now load the file
Yr, dims, T = cm.load_memmap(ca_name_new)
images = np.reshape(Yr.T, [T] + list(dims), order='F') 
    #load frames in python format (T x X x Y)

In [8]:
opts = build_cnmf_params(fr=3.43, decay_time=10, gSig=[14, 12.5, 5])
cnm = run_cnmf(images, opts, dview=dview, n_processes=n_processes)
cnm.estimates.A.shape  # (n_pixels, n_components)

In setting CNMFParams, non-pathed parameters were used; this is deprecated. In some future version of Caiman, allow_legacy will default to False (and eventually will be removed)


(425984, 40)

## Evaluate component quality

Runs the SNR / spatial-correlation / CNN checks against the thresholds set in `build_cnmf_params` (`min_SNR`, `rval_thr`, `min_cnn_thr`) and populates `idx_components` (accepted) / `idx_components_bad` (rejected).

In [9]:
cnm.estimates.evaluate_components(images, cnm.params, dview=dview)
print(f"Accepted: {len(cnm.estimates.idx_components)}")
print(f"Rejected: {len(cnm.estimates.idx_components_bad)}")

Accepted: 29
Rejected: 11


## Interactive 3D viewer

`image_type` can be 'mean', 'max', or 'corr' (the latter needs `Yr`, the raw movie reshaped to pixels x frames, to compute the correlation image -- skip it for now). `max_projection=True` collapses the chosen `axis` instead of scrolling through z-slices.

In [10]:
cnm.estimates.nb_view_components_3d(dims=cnm.estimates.dims, image_type='mean')

ValueError: axes don't match array

## Manual curation methods

`select_components` keeps only the given indices; with `save_discarded_components=True` the rest are stashed on `estimates.discarded_components` rather than deleted, so `restore_discarded_components()` can bring them back. This is the non-destructive accept/reject mechanism a custom adjustment tool would drive.

In [ ]:
# Keep only the components evaluate_components accepted
cnm.estimates.select_components(use_object=True, save_discarded_components=True)
cnm.estimates.A.shape

In [ ]:
# Undo -- brings the rejected components back
cnm.estimates.restore_discarded_components()
cnm.estimates.A.shape

## Save / reload round trip

`cnm.save(...)` writes the entire object (estimates, params, dims) to one HDF5 file; `load_CNMF(...)` reconstructs it fully -- this is the supported way to persist results between sessions.

In [ ]:
results_path = output_dir / 'cnmf_results.hdf5'
cnm.save(str(results_path))
results_path

In [ ]:
cnm_reloaded = load_CNMF(str(results_path), dview=dview)
cnm_reloaded.estimates.A.shape

In [ ]:
cm.stop_server(dview=dview)